## Imports

In [1]:
import sys
sys.path.append("..")

from sentence_transformers import SentenceTransformer
import pyarrow.parquet as pq
import numpy as np
import pandas as pd
import faiss
import pickle

from src.rag import (
    load_vector_store,
    load_generator,
    ask_question
)
from src.streaming import build_streaming_index
from src.rag_helper import retrieve

## Load Embeddings

### Build the index by streaming

In [2]:
# build_streaming_index(
#     "../vector_store/complaint_embeddings.parquet"
# )

### Load the index

In [3]:
index = faiss.read_index("complaints.index")

with open("row_mapping.pkl", "rb") as f:
    row_mapping = pickle.load(f)

### Retrieve only the rows you need

In [4]:
parquet = pq.ParquetFile("../vector_store/complaint_embeddings.parquet")

## Load Embedding Mode

In [5]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

## Load LLM

In [6]:
generator = load_generator()

Device set to use cpu


## Prompt Builder

In [7]:
def build_prompt(question, retrieved_rows):

    context = "\n\n".join([r["document"] for r in retrieved_rows])

    prompt = f"""
You are a financial complaint analyst for CrediTrust.

Use ONLY the context below to answer.

If the answer is not present, say you don't have enough information.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

## Full RAG Function

In [8]:
def ask_question(question):

    retrieved = retrieve(
        question,
        embedding_model,
        index,
        row_mapping,
        parquet,
        k=5
    )

    prompt = build_prompt(question, retrieved)

    response = generator(
        prompt,
        max_new_tokens=256,
        do_sample=False
    )

    return response[0]["generated_text"], retrieved

## TEST: Single Question

In [9]:
question = "Why are customers unhappy with credit cards?"

answer, retrieved = ask_question(question)

print(answer)

they don't understand the features and benefits they offer


## Show Retrieved Evidence

In [10]:
for r in retrieved:
    print("="*80)
    print(r["document"][:500])

card company and was very unhappy and frustrated. as a consumer i feel that we apply for new credit cards because of the features and benefits they offer, however we need to understand how to use them. i am not happy with the customer service and i am not happy with the misinformation i was given. i have been given misinformation by several customer service representatives and i feel that the credit card company is taking advantage of consumers who dont understand the features and benefits.
creditors. i have an exceptional payment history. there was no reason for them to reduce my credit limit at all doing so caused me harm by making my credit usage shoot up. what the is up with these companies it like here is the card but don't use it!
credit card companies think of their card holders. the indifferent response to a long-term card holder again came to me as a shock given how this particular company was receptive to me in the past when i called about other matters. i was obviously stupi

## Evaluation Setup

In [11]:
questions = [
    "Why are customers unhappy with credit cards?",
    "What problems occur during money transfers?",
    "Why are personal loan customers complaining?",
    "What are the most common savings account issues?",
    "Why are duplicate charges happening?",
    "What fraud complaints are common?",
    "Why are transactions delayed?"
]

## Run Evaluation

In [12]:
results = []

for q in questions:

    answer, retrieved = ask_question(q)

    results.append({
        "Question": q,
        "Generated Answer": answer,
        "Retrieved Sources": "\n---\n".join([r["id"] for r in retrieved[:2]]),
        "Quality Score": "",
        "Comments": ""
    })

## Evaluation Table

In [13]:
evaluation = pd.DataFrame(results)
evaluation

,Question,Generated Answer,Retrieved Sources,Quality Score,Comments
0,Why are customers unhappy with credit cards?,they don't understand the features and benefit...,3509835_2\n---\n8892605_1,,
1,What problems occur during money transfers?,"fraudulent transfer, dollars. , multiple fraud...",11520794_0\n---\n11672115_0,,
2,Why are personal loan customers complaining?,They are being harassed by a loan shark.,2337649_1\n---\n4719303_2,,
3,What are the most common savings account issues?,"much money in the savings account as possible,...",8356367_2\n---\n2893337_1,,
4,Why are duplicate charges happening?,a merchant whose point of sale terminal seems ...,11383498_3\n---\n7624705_1,,
5,What fraud complaints are common?,"emotional coercion, soulmate scams, billing co...",3622189_8\n---\n10290501_2,,
6,Why are transactions delayed?,to extract unauthorized interest,7541829_0\n---\n8219260_12,,
